# Figure 4 — Cross-Pathway Shield Coordination

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/figures/figure4_perturbation.ipynb)

Reproduces **Figure 4** from Çavuş & Kuşkucu (2026): pseudo-perturbation reveals mutual reinforcement among defence pathways, and TBXT is the single most tightly locked gene in the CSC population.

**Panels**

| Panel | What it shows | msep function |
|---|---|---|
| A | Heatmap: effect of high expression of each shield gene on the CV of other pathways | `msep.perturbation.pseudo_perturbation` |
| B | Schematic of mutual reinforcement among defence programs | illustrative (drawn in notebook) |
| C | Gene-level CV for 15 key defence genes, coloured by pathway | `result.gene_cv` |

## 1. Install

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Load dataset and run perturbation analysis

Uses the bundled demo. Swap in the Arrieta 2025 cohort for paper-exact numbers (see Figure 2 notebook, Section 10).

In [ ]:
import msep
import pandas as pd
import matplotlib.pyplot as plt

adata = msep.datasets.load_example(n_cells=500, seed=42)
result = msep.profile(
    adata,
    pathways='cancer_defense',
    cell_type_key='cell_type',
    compute_bootstrap=False,
    compute_gene_cv=True,
    compute_perturbation=True,
    shield_genes=['VIM', 'TBXT', 'GPX4', 'FTH1', 'HLA-E', 'B2M'],
    perturbation_cell_type='CSC_TBXT+',
    n_perm=500,
)
result

## 3. Panel A — Pseudo-perturbation heatmap

Each row is a shield gene; each column is a target pathway. Cell colour encodes the ΔCV between high-expression and low-expression CSC (blue = decreased CV = tightened coordination). Asterisks mark permutation p < 0.05.

In [ ]:
import seaborn as sns

perturb = result.perturbation.copy()
print('Perturbation results:')
print(perturb)

delta = perturb.pivot(index='shield_gene', columns='target_pathway', values='delta_cv')
pvals = perturb.pivot(index='shield_gene', columns='target_pathway', values='perm_p')

fig_a, ax_a = plt.subplots(figsize=(6, 5))
annot = delta.copy().astype(str)
for i in delta.index:
    for j in delta.columns:
        val = delta.loc[i, j]
        p = pvals.loc[i, j]
        star = '*' if p < 0.05 else ''
        annot.loc[i, j] = f'{val:+.2f}{star}'

sns.heatmap(delta, annot=annot, fmt='', cmap='RdBu_r', center=0,
            linewidths=0.5, cbar_kws={'label': 'delta CV (high - low)'}, ax=ax_a)
ax_a.set_title('A - Pseudo-perturbation: delta CV by shield gene x target pathway\n(asterisks: permutation p < 0.05)')
fig_a.tight_layout()
fig_a.savefig('figure4_panelA_perturbation.pdf', bbox_inches='tight')
fig_a.savefig('figure4_panelA_perturbation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
key_genes = {
    'TBXT': 'emt', 'VIM': 'emt', 'FN1': 'emt', 'CDH1': 'emt', 'KRT18': 'emt',
    'SMAD2': 'emt', 'SMAD3': 'emt',
    'GPX4': 'ferroptosis', 'SLC7A11': 'ferroptosis', 'FTH1': 'ferroptosis', 'ACSL4': 'ferroptosis',
    'HLA-E': 'immune_evasion', 'B2M': 'immune_evasion', 'CD274': 'immune_evasion', 'MICA': 'immune_evasion',
}
pathway_colors = {'emt': '#2ECC71', 'ferroptosis': '#E74C3C', 'immune_evasion': '#3498DB'}

rows = []
for pw, gene_cv_df in result.gene_cv.items():
    if gene_cv_df is None or gene_cv_df.empty:
        continue
    sub = gene_cv_df[gene_cv_df['gene'].isin(key_genes)].copy()
    sub['pathway'] = pw
    rows.append(sub)
gene_cv_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
gene_cv_csc = gene_cv_all.sort_values('cv')

fig_c, ax_c = plt.subplots(figsize=(9, 5))
colors = [pathway_colors[key_genes[g]] for g in gene_cv_csc['gene']]
ax_c.barh(gene_cv_csc['gene'], gene_cv_csc['cv'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(gene_cv_csc.iterrows()):
    ax_c.text(row['cv'] + 0.05, i, f'{row["cv"]:.2f}', va='center', fontsize=8)

handles = [plt.Rectangle((0,0),1,1, color=c) for c in pathway_colors.values()]
ax_c.legend(handles, pathway_colors.keys(), loc='lower right', title='Pathway')
ax_c.set_xlabel('Gene-level CV in CSC')
ax_c.set_title('C · Gene-level CV: key defence genes, coloured by pathway')
ax_c.spines[['top', 'right']].set_visible(False)
fig_c.tight_layout()
fig_c.savefig('figure4_panelC_gene_cv.pdf', bbox_inches='tight')
fig_c.savefig('figure4_panelC_gene_cv.png', dpi=300, bbox_inches='tight')
plt.show()

## Panel B — Mutual-reinforcement schematic

Panel B in the paper is an illustrative schematic (three interlinked shields representing EMT, ferroptosis, and immune-evasion, with double-headed arrows showing mutual reinforcement). Reproduce it in your preferred vector-graphics tool from the underlying perturbation pattern printed by Panel A — positive evidence for an A→B arrow is any (shield_gene_A, target_pathway_B) cell where ΔCV < 0 and p < 0.05.